In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from TIF import Info, Anotaciones

In [104]:
# canales = [i +1 for i in range(3)]
canales = ["F2", "F3"]
tipos_canales = ["ecg"] * len(canales)
sujeto = {"Nombre": "Juan Acosta",
          "Edad": 26}

In [105]:
infoo = Info(experimenter="Juan Acosta", subject_info=sujeto, ch_names=canales,
            ch_types=tipos_canales, bads=None, description="Pruebita", fm=512)
infoo.keys()
infoo.data["Nombre canales"]
type(infoo.get("Nombre canales"))

list

In [70]:
inicio = [5.0, 12.5, 20.0]
duracion = [2.0, 3.0, 3.5]
descripcion = ['Inicio_Experimento', 'Evento_1', 'Evento_2']

In [71]:
anotacioness = Anotaciones(onset=inicio, duration=duracion, description=descripcion)

In [ ]:
class RawSignal:
    """
    Clase para manejar señales fisiológicas en formato NumPy.
    Este constructor permite inicializar el objeto 'RawSignal' a partir de un array de datos ,
    con información adicional de los canales y el índice de la primera muestra."""
    
    def __init__(self, data:np.ndarray, sfreq:float, info:Info=None, anotaciones:Anotaciones=None, first_samp:int=0):
        """
        Inicializa una instancia de la clase RawSignal.
        
        Args:
            data : Matriz de datos con forma '(n_canales , n_muestras)'.
            sfreq : Frecuencia de muestreo de la señal en Hz.
            info : Por defecto es None. Información adicional sobre la señal. El diccionario contiene info relevante de la señal
            anotaciones : Objeto de tipo Anotaciones que almacena eventos asociados a la señal y al experimento.
            first_samp : Índice del primer muestreo a utilizar (default es 0).
            
        Raises:
            ValueError : Si el array 'data' no tiene la forma '(n_canales , n_muestras)'.
            ValueError : Si el índice 'first_samp' está fuera del rango de la señal. """
        
        if not isinstance(data, np.ndarray):     # Que data sea un array de NumPy
            raise ValueError("El parámetro 'data' debe ser un array de NumPy (np.ndarray).")

        if data.ndim != 2:                       # Que data tenga dos dimensiones 
            raise ValueError("El array 'data' debe tener dos dimensiones: (n_canales, n_muestras).")

        n_muestras = data.shape[1]               # Número de muestras es la segunda dimensión
        if not (0 <= first_samp < n_muestras):   # Que first_samp sea un entero positivo y menor que el numero de muestras
            raise ValueError("El índice 'first_samp' está fuera del rango de muestras disponibles.")

        self.data = data
        self.sfreq = sfreq
        self.info = info
        self.anotaciones = anotaciones
        self.first_samp = first_samp
        
    def get_data(self, picks=None, start:float|int=0, stop:float|int=0, reject:float=None, times:bool=False):
        """
        Obtiene muestras de la señal en un rango dado.
        Args:
            picks (str o array_like) : Canales o índices a extraer. Si es 'None', se seleccionan todos los canales.
            start : Tiempo inicial (en segundos) para extraer muestras (por defecto 0).
            stop : Tiempo final (en segundos) para extraer muestras (por defecto 0, que significa hasta el final de la señal).
            reject : Valor pico a pico de umbral para rechazar canales. Si una muestra supera este umbral, el canal se descarta (por defecto 'None').
            times : Si es 'True', se retorna también el vector de tiempos asociado a las muestras.

        Returns:
            np.ndarray : Matriz con los datos seleccionados (n_canales x n_muestras).
            np.ndarray (opcional) : Vector de tiempos (solo si 'times=True').

        Raises
            ValueError : Si los índices seleccionados están fuera de rango. """
        
        n_canales, n_muestras = self.data.shape                       # Número de canales y muestras  
    
        start_idx = int(start * self.sfreq)                           # Convertir start de segundos a número de muestra
        stop_idx = int(stop * self.sfreq) if stop > 0 else n_muestras # Si stop es mayor que 0, convertir a indice de muestra, sino, tomar hasta el final

        if not (0 <= start_idx < stop_idx <= n_muestras):             # Si no se cumple que start_idx es mayor o igual a 0 y menor que stop_idx y ademas stop_idx es menor o igual al numero de muestras, lleva al error
            raise ValueError("Índices de tiempo fuera de rango.")

        # Seleccionar canales
        if picks is None:                             # Si picks queda por defecto, seleccionar todos los canales
            canales_idx = np.arange(n_canales)

        elif isinstance(picks, (list, np.ndarray)):
            if all(isinstance(pick, int) for pick in picks):   # Verificar que los canales ingresados sean todos enteros
                if np.isin(picks, np.arange(n_canales)).all(): # Verificar si los canales ingresados existen
                    canales_idx = np.array(picks)
                else:
                    raise ValueError("Error: algunos canales ingresados no existen")
            
            elif all(isinstance(pick, str) for pick in picks):   # Verificar que los canales ingresados sean todos strings
                if self.info is None:
                    raise ValueError("Debe ingresar el objeto Info")
                
                nombres = self.info.get("Nombre canales")       # Aplicamos el médoto de Info para obtener los canales del diccionario
                try:
                    canales_idx = [nombres.index(canal) for canal in picks] # Si "canal" esta en los canels pasados desde Info, obtiene el indice y se agrega a canales_idx
                except ValueError:
                    raise ValueError(f"Error: uno o más canales no se encuentran en los canales Ingresados desde el objeto Info")
            else:
                raise ValueError("Error: los canales ingresados no tienen el mismo formato")

        else:
            raise ValueError("Formato de 'picks' inválido. Deben ser strings o enteros.")

        datos = self.data[np.array(canales_idx), start_idx : stop_idx]  # Extraer datos de los canales

        if reject is not None:                 # Aplicar umbral de rechazo si se especifica
            pico_pico = np.ptp(datos, axis=1)  # Toma el max y min del canal y los resta, obteniendo el valor pico_pico de cada canal (axis=1 filas)
            filtro = pico_pico < reject        # Filtro para quedarme solo con los valores pico_pico menores a reject
            datos = datos[filtro]

        if times is True:
            tiempo_vector = np.arange(start_idx, stop_idx) / self.sfreq # t = muestra/fm
            return datos, tiempo_vector

        return datos
    
    def drop_channels(self, ch_names) -> "RawSignal":
        """
        Elimina uno o más canales a partir de ch_names
        Parameters
        ----------
        ch_names:array_like
        Nombres de canales a eliminar
        Returns
        ----------
        RawSignal
        """

        # Validaciones
        if self.info is None or "canales" not in self.info:
            raise ValueError("No se puede eliminar canales sin info['canales'].")

        if isinstance(ch_names, str):
            ch_names = [ch_names]
        elif not isinstance(ch_names, (list, np.ndarray)):
            raise ValueError("'ch_names' debe ser una lista, array o string.")

        # Lista original de canales
        canales_actuales = self.info["canales"]

        # Verificar que todos los canales existan
        for ch in ch_names:
            if ch not in canales_actuales:
                raise ValueError(f"El canal '{ch}' no existe en info['canales'].")

        # Crear máscara para mantener los canales no eliminados
        canales_a_mantener = [i for i, nombre in enumerate(canales_actuales) if nombre not in ch_names]

        # Filtrar data y actualizar info
        nueva_data = self.data[canales_a_mantener, :]
        nueva_info = self.info.copy()
        nueva_info["canales"] = [canales_actuales[i] for i in canales_a_mantener]

        # Crear y devolver nueva instancia de RawSignal
        return RawSignal(
            data=nueva_data,
            sfreq=self.sfreq,
            info=nueva_info,
            anotaciones=self.anotaciones,
            first_samp=self.first_samp
        )
        
    def crop(self, tmin=0.0, tmax=None)->"RawSignal":
        """
        Obtiene un trozo (Crop) de RawSignal. Limita los datos dentro de RawSignal
        para obtener un nuevo objeto RawSignal pero con una cantidad de muestras recortadas.
        El parámetro 'first_samp' se configura adecuadamente.
        
        Parameters
        ------------
        tmin : float , optional
            Tiempo inicial , en segundos , para iniciar el recorte (por defecto es 0.0).
        tmax : float or None , optional
            Tiempo final , en segundos , para finalizar el recorte (por defecto es None).
        
        Returns
        -----------
        RawSignal
            Nueva instancia de 'RawSignal' que contiene el segmento temporal recortado.
        
        Raises
        -----------
        Value Error
            Si los tiempos 'tmin' o 'tmax' están fuera del rango de la señal.
        """
        
        n_canales, n_muestras = self.data.shape
        duracion_total = n_muestras / self.sfreq

        #Validaciones
        if not isinstance(tmin, (int, float)) or tmin < 0: ## si tmin no es entero, flotante o positivo
            raise ValueError("'tmin' debe ser un número positivo.")
        if tmax is not None and (not isinstance(tmax, (int, float)) or tmax <= tmin): #si tmax no fue definido por 
                                                                                #el usuario Y ADEMAS, no es un 
                                                                                # numero o ese numero es mayor o 
                                                                                # igual a tmin...
            raise ValueError("'tmax' debe ser mayor que 'tmin'.")

        if tmin > duracion_total: ##Chequeo si tmin es mayor que la duracion total de la señal 
            raise ValueError(f"'tmin' está fuera del rango de la señal ({duracion_total:.2f} s).")
        if tmax is not None and tmax > duracion_total: ##si tmax fue  definido por el usuario 
                                                         #y es mayor que la duracion total de la señal
            raise ValueError(f"'tmax' está fuera del rango de la señal ({duracion_total:.2f} s).")

        # Convertir a índices
        start_idx = int(tmin * self.sfreq)
        end_idx = int(tmax * self.sfreq) if tmax is not None else n_muestras

        # Extraer el segmento de la señal
        datos_crop = self.data[:, start_idx:end_idx]

        # Ajustar first_samp
        new_first_samp = self.first_samp + start_idx

        # Crear y retornar nueva instancia
        return RawSignal(
            data=datos_crop,
            sfreq=self.sfreq,
            info=self.info,
            anotaciones=self.anotaciones,
            first_samp=new_first_samp
        )

    def describe(self):
        pass

    def filter(self):
        pass

    def pick(self):
        pass

    def set_anotaciones(self):
        pass

    def plt(self):
        pass     

In [46]:
eeg_data = np.load("../2. tests/eeg/eeg_signal.npy")

In [47]:
eeg_data.shape

(62, 388047)

In [111]:
# Generar el objeto RawSignal
raw_signal = RawSignal(data=eeg_data, sfreq=512, info=infoo, anotaciones=anotacioness)

In [172]:
# Obtener muestas
# muestras = raw_signal.get_data(start=0, stop=1)
muestras, vector = raw_signal.get_data(picks=[0,1,2], start=0, stop=10, times=True, reject=30000)
print(muestras.shape)
print(vector)

# muestras = raw_signal.get_data(picks=["F2", "F3"], start=0, stop=1)  # Este no se si anda bien
# print(muestras)
# print(muestras.shape)

(2, 5120)
[0.00000000e+00 1.95312500e-03 3.90625000e-03 ... 9.99414062e+00
 9.99609375e+00 9.99804688e+00]
